In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt
import os
import matplotlib.dates as mdates
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objs as go
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv('C:/Users/Asifuzzaman.jishan/Downloads/ENTSO-E_Data_analysis_and_ml/Data/expected_csv_data.csv')

In [ ]:
df.head(5)

In [ ]:
df.tail(5)

In [ ]:
df.describe

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.isna().sum()

In [ ]:
df.info()

In [ ]:
df = df.drop(['Biomass', 'Fossil Brown coal/Lignite', 'Fossil Gas', 'Fossil Hard coal', 'Hydro Pumped Storage.1', 
              'Hydro Run-of-river and poundage', 'Hydro Water Reservoir', 'Nuclear', 'Other', 'Other renewable', 'Solar', 'Solar.1', 'Waste', 'Wind Offshore', 
              'Wind Onshore', 'Wind Onshore.1'], axis=1)

In [ ]:
df.isna().sum()

In [ ]:
# Check for missing values in the DataFrame
print(df.isna().sum())

In [ ]:
# Drop rows with missing values
df = df.dropna()

# Confirm the missing value has been removed
print(df.isna().sum())

In [ ]:
df.info()

In [ ]:
df['Unnamed: 0'] = pd.to_datetime(df['Unnamed: 0'], errors='coerce', utc=True)
df = df.dropna(subset=['Unnamed: 0'])

In [ ]:
df.info()

In [ ]:
# List of columns you want to convert
columns_to_convert = ['Fossil Oil', 'Hydro Pumped Storage', 'Geothermal']

# Loop through each column in the DataFrame
for column in columns_to_convert:
    # Convert the column to numeric, coerce any errors to NaN
    df[column] = pd.to_numeric(df[column], errors='coerce')

    # If you have NaNs after coercion and want to fill them with 0
    df[column] = df[column].fillna(0).astype(int)

In [ ]:
df.info()

In [ ]:
df.head(10)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.isna().sum()

In [ ]:
df.info()

In [ ]:
# Convert the 'Unnamed: 0' column to datetime
df['Unnamed: 0'] = pd.to_datetime(df['Unnamed: 0'])

In [ ]:
# Set the datetime column as the index of the DataFrame
df.set_index('Unnamed: 0', inplace=True)

In [ ]:
#path = 'C:/Users/Asifuzzaman.jishan/Downloads/ENTSO-E_Data_analysis_and_ml/Graph_2_features'
#os.mkdir(path)

#### K-Means

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import pandas as pd

# Define the sample size
sample_size = 500  # Adjust sample size as needed

# Select the features for clustering
features = ['Fossil Oil', 'Geothermal', 'Hydro Pumped Storage']  # Columns to use for clustering
df_features = df[features]

# Take a random sample of the data
df_sampled = df_features.sample(n=sample_size, random_state=42)

# Normalize the features using MinMaxScaler
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_sampled)

# Define range of clusters
K = range(2, 11)  # Adjust range as needed

# Perform K-Means clustering and evaluate
silhouette_scores = []
for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(scaled_data)
    silhouette_scores.append(silhouette_score(scaled_data, kmeans.labels_))

# Print silhouette scores for each cluster count
print("\nSilhouette Scores for Different Cluster Counts:")
for k, score in zip(K, silhouette_scores):
    print(f"  Number of clusters: {k}, Silhouette Score: {score:.4f}")

In [ ]:
# Plot Silhouette Scores for K-Means
plt.figure(figsize=(10, 6))
plt.plot(K, silhouette_scores, marker='o', linestyle='--', label='K-Means Silhouette Score')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Scores for K-Means Clustering')
plt.legend()
plt.show()

In [ ]:
# Best number of clusters based on highest silhouette score
best_k = K[np.argmax(silhouette_scores)]
print(f"\nBest number of clusters: {best_k}")

# Apply KMeans with the best number of clusters
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(scaled_data)

# Add cluster labels to the sampled DataFrame
df_sampled['Cluster'] = cluster_labels

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Visualization for 3 features in a 3D plot
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# 3D scatter plot
sc = ax.scatter(
    df_sampled['Fossil Oil'], 
    df_sampled['Geothermal'], 
    df_sampled['Hydro Pumped Storage'], 
    c=df_sampled['Cluster'], 
    cmap='viridis', 
    s=50
)

# Set axis labels
ax.set_xlabel('Fossil Oil')
ax.set_ylabel('Geothermal')
ax.set_zlabel('Hydro Pumped Storage')
ax.set_title(f'KMeans Clustering with {best_k} Clusters')

# Add a color bar
plt.colorbar(sc, label='Cluster')
plt.show()


#### K-Medoids

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import pandas as pd

# Define the sample size
sample_size = 500  # Adjust sample size as needed

# Select the features for clustering
features = ['Fossil Oil', 'Geothermal', 'Hydro Pumped Storage']  # Columns to use for clustering
df_features = df[features]

# Take a random sample of the data
df_sampled = df_features.sample(n=sample_size, random_state=42)

# Normalize the features using MinMaxScaler
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_sampled)

# Define range of clusters
K = range(2, 11)  # Adjust range as needed

# Perform K-Medoids clustering and evaluate
silhouette_scores = []
for k in K:
    kmedoids = KMedoids(n_clusters=k, random_state=42, method='pam')
    kmedoids.fit(scaled_data)
    silhouette_scores.append(silhouette_score(scaled_data, kmedoids.labels_))

# Print silhouette scores for each cluster count
print("\nSilhouette Scores for Different Cluster Counts:")
for k, score in zip(K, silhouette_scores):
    print(f"  Number of clusters: {k}, Silhouette Score: {score:.4f}")

# Plot Silhouette Scores for K-Medoids
plt.figure(figsize=(10, 6))
plt.plot(K, silhouette_scores, marker='o', linestyle='--', label='K-Medoids Silhouette Score')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Scores for K-Medoids Clustering')
plt.legend()
plt.show()

In [ ]:
# Choose the best number of clusters based on Silhouette Score
best_k = K[np.argmax(silhouette_scores)]
print(f'Best number of clusters for K-Medoids: {best_k}')

# Apply K-Medoids with the best number of clusters
kmedoids = KMedoids(n_clusters=best_k, random_state=42, method='pam')
kmedoids_labels = kmedoids.fit_predict(scaled_data)

# Add cluster labels to the sampled data
df_sampled = df_features.sample(n=500, random_state=42).copy()  # Ensure we use the same sampled data
df_sampled['Cluster_KMedoids'] = kmedoids_labels

# Evaluate the clustering performance using silhouette score
silhouette_score_kmedoids = silhouette_score(scaled_data, kmedoids_labels)
print(f"Silhouette Score for K-Medoids with {best_k} clusters: {silhouette_score_kmedoids}")

# Visualize the clusters in 2D
plt.figure(figsize=(10, 6))

# Scatter plot with color for clusters and size for 'Hydro Pumped Storage'
plt.scatter(
    df_sampled['Fossil Oil'],
    df_sampled['Geothermal'],
    c=df_sampled['Cluster_KMedoids'],
    s=df_sampled['Hydro Pumped Storage'] * 10,  # Adjust size for better visualization
    cmap='plasma',
    alpha=0.6
)

plt.xlabel('Fossil Oil')
plt.ylabel('Geothermal')
plt.title(f'K-Medoids Clustering with {best_k} Clusters (Size: Hydro Pumped Storage)')
plt.colorbar(label='Cluster')
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Choose the best number of clusters based on Silhouette Score
best_k = K[np.argmax(silhouette_scores)]
print(f'Best number of clusters for K-Medoids: {best_k}')

# Apply K-Medoids with the best number of clusters
kmedoids = KMedoids(n_clusters=best_k, random_state=42, method='pam')
kmedoids_labels = kmedoids.fit_predict(scaled_data)

# Create a copy of the sampled data for visualization and add cluster labels
df_sampled = df_features.sample(n=500, random_state=42).copy()
df_sampled['Cluster_KMedoids'] = kmedoids_labels

# Evaluate the clustering performance using silhouette score
silhouette_score_kmedoids = silhouette_score(scaled_data, kmedoids_labels)
print(f"Silhouette Score for K-Medoids with {best_k} clusters: {silhouette_score_kmedoids}")

# Visualize the clusters in 3D
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# 3D scatter plot with the additional feature 'Hydro Pumped Storage'
sc = ax.scatter(
    df_sampled['Fossil Oil'],
    df_sampled['Geothermal'],
    df_sampled['Hydro Pumped Storage'],
    c=df_sampled['Cluster_KMedoids'],
    cmap='plasma',
    alpha=0.6
)

# Set axis labels
ax.set_xlabel('Fossil Oil')
ax.set_ylabel('Geothermal')
ax.set_zlabel('Hydro Pumped Storage')
ax.set_title(f'K-Medoids Clustering with {best_k} Clusters')

# Add a color bar
plt.colorbar(sc, label='Cluster')
plt.show()